In [1]:
import os
import random

base_path = "/content/drive/MyDrive/House_Room_Dataset"

# Collect all images
image_paths = []

for folder in os.listdir(base_path):
    folder_path = os.path.join(base_path, folder)
    if os.path.isdir(folder_path):
        for img in os.listdir(folder_path):
            image_paths.append(os.path.join(folder_path, img))

print("Total images:", len(image_paths))

Total images: 3093


In [3]:
import pandas as pd

df = pd.read_csv("/content/drive/MyDrive/House_Room_Dataset/House Price Prediction Dataset.csv")

# Assign random image to each row
df["image_path"] = [random.choice(image_paths) for _ in range(len(df))]

df.head()

,Id,Area,Bedrooms,Bathrooms,Floors,YearBuilt,Location,Condition,Garage,Price,image_path
0,1,1360,5,4,3,1970,Downtown,Excellent,No,149919,/content/drive/MyDrive/House_Room_Dataset/Livi...
1,2,4272,5,4,3,1958,Downtown,Excellent,No,424998,/content/drive/MyDrive/House_Room_Dataset/Livi...
2,3,3592,2,2,3,1938,Downtown,Good,No,266746,/content/drive/MyDrive/House_Room_Dataset/Livi...
3,4,966,4,2,2,1902,Suburban,Fair,Yes,244020,/content/drive/MyDrive/House_Room_Dataset/Bedr...
4,5,4926,1,4,2,1975,Downtown,Fair,Yes,636056,/content/drive/MyDrive/House_Room_Dataset/Livi...


In [4]:
import numpy as np
from tensorflow.keras.preprocessing.image import load_img, img_to_array

IMG_SIZE = (128, 128)

def load_image(path):
    img = load_img(path, target_size=IMG_SIZE)
    img = img_to_array(img) / 255.0
    return img

images = np.array([load_image(p) for p in df["image_path"]])

In [16]:
X_tab = df.drop(columns=["Price", "Id", "image_path"])
y = df["Price"]

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

# define column types
num_cols = X_tab.select_dtypes(include=['int64', 'float64']).columns
cat_cols = X_tab.select_dtypes(include=['object']).columns

# preprocessing pipeline
preprocessor = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown='ignore'), cat_cols)
])

X_tab_scaled = preprocessor.fit_transform(X_tab)

In [17]:
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Input
from tensorflow.keras.models import Model

image_input = Input(shape=(128,128,3))

x = Conv2D(32, (3,3), activation='relu')(image_input)
x = MaxPooling2D()(x)

x = Conv2D(64, (3,3), activation='relu')(x)
x = MaxPooling2D()(x)

x = Flatten()(x)
x = Dense(64, activation='relu')(x)

In [18]:
tabular_input = Input(shape=(X_tab_scaled.shape[1],))

y_tab = Dense(32, activation='relu')(tabular_input)
y_tab = Dense(16, activation='relu')(y_tab)

In [19]:
from tensorflow.keras.layers import concatenate

combined = concatenate([x, y_tab])

z = Dense(64, activation='relu')(combined)
z = Dense(1)(z)

In [20]:
model = Model(inputs=[image_input, tabular_input], outputs=z)

model.compile(
    optimizer='adam',
    loss='mse',
    metrics=['mae']
)

In [26]:
model.fit(
    [images, X_tab_scaled],
    y,
    epochs=10,
    batch_size=100,
    validation_split=0.2
)

Epoch 1/10
16/16 ━━━━━━━━━━━━━━━━━━━━ 41s 3s/step - loss: 76708937728.0000 - mae: 238842.7812 - val_loss: 81692917760.0000 - val_mae: 247489.5938
Epoch 2/10
16/16 ━━━━━━━━━━━━━━━━━━━━ 83s 3s/step - loss: 77269565440.0000 - mae: 239586.0000 - val_loss: 81209688064.0000 - val_mae: 246983.7969
Epoch 3/10
16/16 ━━━━━━━━━━━━━━━━━━━━ 39s 2s/step - loss: 77256523776.0000 - mae: 239214.0625 - val_loss: 81767481344.0000 - val_mae: 247555.8750
Epoch 4/10
16/16 ━━━━━━━━━━━━━━━━━━━━ 41s 3s/step - loss: 77056155648.0000 - mae: 238636.2344 - val_loss: 80723345408.0000 - val_mae: 246520.1406
Epoch 5/10
16/16 ━━━━━━━━━━━━━━━━━━━━ 41s 3s/step - loss: 76721586176.0000 - mae: 238750.5781 - val_loss: 79597543424.0000 - val_mae: 245091.1562
Epoch 6/10
16/16 ━━━━━━━━━━━━━━━━━━━━ 39s 2s/step - loss: 76665544704.0000 - mae: 238653.9062 - val_loss: 79523250176.0000 - val_mae: 244933.3594
Epoch 7/10
16/16 ━━━━━━━━━━━━━━━━━━━━ 43s 3s/step - loss: 76652290048.0000 - mae: 238781.6250 - val_loss: 80131670016.0000 -

In [27]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

preds = model.predict([images, X_tab_scaled])

mae = mean_absolute_error(y, preds)
rmse = np.sqrt(mean_squared_error(y, preds))

print("MAE:", mae)
print("RMSE:", rmse)

63/63 ━━━━━━━━━━━━━━━━━━━━ 13s 210ms/step
MAE: 240013.328125
RMSE: 277879.71613631677
